<a href="https://colab.research.google.com/github/khushi-eng/DMV/blob/main/IR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install nltk pgmpy nltk scikit-learn pandas

In [ ]:
#Prac1
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# Sample text document
file_path = r"C:\Users\khush\Downloads\IR_pracs\sample.txt"
with open(file_path, "r", encoding="utf-8") as file:
  text = file.read()

print("Original Text:")
print(text)

# 1. Lowercasing and Tokenization
tokens = word_tokenize(text.lower())
print("\nTokens:")
print(tokens)

# 2. Stop Word & Punctuation Removal
stop_words = set(stopwords.words('english'))
filtered_words = [
    word for word in tokens if word.isalpha() and word not in stop_words
]
print("\nAfter Stop Word Removal:")
print(filtered_words)

# 3. Stemming
stemmer = PorterStemmer()
stemmed_words = [stemmer.stem(word) for word in filtered_words]
print("\nAfter Stemming:")
print(stemmed_words)

In [ ]:
#Prac2
from collections import defaultdict
import os
import re

# Folder path containing your .txt files
folder_path = r"C:\Users\khush\Downloads\IR_pracs"
file_names = ["doc1.txt", "doc2.txt", "doc3.txt"]

# 1. Read files and build the Inverted Index
inverted_index = defaultdict(list)

print("Documents:")
for fname in file_names:
  full_path = os.path.join(folder_path, fname)
  with open(full_path, "r", encoding="utf-8") as f:
    content = f.read().strip()
    print(f"{fname}: {content}")

    # Convert to lowercase and tokenize words
    words = re.findall(r"\b[a-z]+\b", content.lower())

    # Store file name for each unique word
    for word in set(words):
      inverted_index[word].append(fname)

print("\n----- INVERTED INDEX -----")
for word in sorted(inverted_index):
  print(f"{word:15} -> {inverted_index[word]}")


# 2. Search / Retrieval Function
def search(query):
  query_words = re.findall(r"\b[a-z]+\b", query.lower())
  if not query_words:
    return []
  result_sets = [set(inverted_index.get(word, [])) for word in query_words]
  return sorted(set.intersection(*result_sets))


# 3. Interactive Search Query
query = input("\nEnter search query: ")
result = search(query)

print("Query:", query)
if result:
  print("Matching Documents:", result)
else:
  print("No matching documents found.")

In [ ]:
#Prac3
import pandas as pd
from pgmpy.inference import VariableElimination
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteMLE

# 1. Load the Heart Disease medical dataset
url = "https://gist.githubusercontent.com/prajwalmani/e4d76fa6f32adc53949c23db1ed037f2/raw/data7_heart.csv"
heartDisease = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Shape:", heartDisease.shape)
print(heartDisease.head())

# 2. Construct the Bayesian Network structure (DAG)
model = DiscreteBayesianNetwork([
    ("age", "heartdisease"),
    ("sex", "heartdisease"),
    ("exang", "heartdisease"),
    ("cp", "heartdisease"),
    ("heartdisease", "restecg"),
    ("heartdisease", "chol"),
])

print("\nBayesian Network created successfully!")
print("Nodes:", model.nodes())
print("Edges:", model.edges())

# 3. Learn Conditional Probability Distributions (CPDs)
estimator = DiscreteMLE()
model.fit(heartDisease, estimator=estimator)

print("\nCPDs learned successfully!")
print("Model Check:", model.check_model())

# 4. Perform Diagnosis using Variable Elimination
inference = VariableElimination(model)

# Query 1: Probability of Heart Disease given Chest Pain type (cp = 2)
print("\nProbability of Heart Disease given cp = 2:")
q1 = inference.query(variables=["heartdisease"], evidence={"cp": 2})
print(q1)

# Query 2: Probability of Heart Disease given Rest ECG (restecg = 1)
print("\nProbability of Heart Disease given restecg = 1:")
q2 = inference.query(variables=["heartdisease"], evidence={"restecg": 1})
print(q2)

In [ ]:

# Practical 4: E-mail Spam Filtering using Text Classification

# 1. Import libraries

import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


# 2. Download stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()


# 3. Load dataset
# Upload the extracted SMSSpamCollection file to Colab.

data = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)

print("First five records:")
print(data.head())

print("\nDataset shape:", data.shape)

print("\nLabel counts:")
print(data["label"].value_counts())


# 4. Convert labels into numbers

data["label"] = data["label"].map({
    "ham": 0,
    "spam": 1
})


# 5. Text preprocessing

def preprocess(text):

    # Convert text to lowercase
    text = str(text).lower()

    # Replace URLs with the word url
    text = re.sub(r'http\S+|www\S+', ' url ', text)

    # Replace numbers with the word number
    text = re.sub(r'\d+', ' number ', text)

    # Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Tokenization
    words = text.split()

    # Remove stopwords
    words = [w for w in words if w not in stop_words]

    # Stemming
    words = [stemmer.stem(w) for w in words]

    return " ".join(words)


data["message"] = data["message"].apply(preprocess)


# 6. Separate input and output

X = data["message"]
y = data["label"]


# 7. Split dataset into training and testing data

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


# 8. TF-IDF feature extraction

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True
)

X_train = tfidf.fit_transform(X_train)
X_test = tfidf.transform(X_test)


# 9. Create and train KNN model

model = KNeighborsClassifier(
    n_neighbors=3,
    metric="cosine",
    weights="distance"
)

model.fit(X_train, y_train)


# 10. Predict results

y_pred = model.predict(X_test)


# 11. Evaluate model

print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Ham", "Spam"],
    zero_division=0
))


# 12. Test your own message

while True:

    message = input("\nEnter a message (type exit to stop): ")

    if message.lower() == "exit":
        break

    # Preprocess message
    message = preprocess(message)

    # Convert message into TF-IDF vector
    message_vector = tfidf.transform([message])

    # Predict message category
    result = model.predict(message_vector)[0]

    if result == 1:
        print("SPAM MESSAGE")
    else:
        print("NOT SPAM (HAM)")


First five records:
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

Dataset shape: (5572, 2)

Label counts:
label
ham     4825
spam     747
Name: count, dtype: int64


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



Training samples: 4457
Testing samples: 1115

Accuracy: 0.9668161434977578

Confusion Matrix:
[[952  14]
 [ 23 126]]

Classification Report:
              precision    recall  f1-score   support

         Ham       0.98      0.99      0.98       966
        Spam       0.90      0.85      0.87       149

    accuracy                           0.97      1115
   macro avg       0.94      0.92      0.93      1115
weighted avg       0.97      0.97      0.97      1115

SPAM MESSAGE
NOT SPAM (HAM)
SPAM MESSAGE


In [ ]:

# Practical 5: Agglomerative Hierarchical Clustering

# 1. Import libraries

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering

import scipy.cluster.hierarchy as shc


# 2. Load dataset

url = "https://raw.githubusercontent.com/j4-metrics/data_science_projects/main/cc_general.csv"

data = pd.read_csv(url)

print("Dataset loaded successfully!")
print(data.head())

print("\nDataset shape:", data.shape)


# 3. Remove customer ID column

data = data.drop("CUST_ID", axis=1)


# 4. Handle missing values

data = data.fillna(data.median())

print("\nMissing values:", data.isnull().sum().sum())


# 5. Standardize the data

scaler = StandardScaler()

X_scaled = scaler.fit_transform(data)

print("\nData standardized successfully.")


# 6. Draw Dendrogram

# Use a sample to make dendrogram faster

sample = X_scaled[:500]

plt.figure(figsize=(10, 5))

shc.dendrogram(
    shc.linkage(sample, method="ward")
)

plt.title("Dendrogram")
plt.xlabel("Data Points")
plt.ylabel("Euclidean Distance")

plt.show()


# 7. Apply Agglomerative Clustering

model = AgglomerativeClustering(
    n_clusters=3,
    linkage="ward"
)

labels = model.fit_predict(X_scaled)


# 8. Display cluster results

data["Cluster"] = labels

print("\nCluster assignments:")
print(data[["Cluster"]].head(10))

print("\nNumber of customers in each cluster:")
print(data["Cluster"].value_counts())


# 9. Visualize clusters using two features

plt.figure(figsize=(7, 5))

plt.scatter(
    data["BALANCE"],
    data["PURCHASES"],
    c=labels,
    cmap="rainbow"
)

plt.xlabel("Balance")
plt.ylabel("Purchases")
plt.title("Agglomerative Hierarchical Clustering")

plt.show()
